# Lab 7: Cùng hàm clean_trades với capstone

Notebook này không viết lại luật làm sạch. Nó import `clean_trades` từ `Data_Lakehouse_lab/pipeline/transforms.py` và chạy trên một mẫu local, bằng Spark `mapInPandas`, cùng cách `silver.py` gọi hàm đó.


In [ ]:

from pathlib import Path
import sys
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType, TimestampType

here = Path.cwd().resolve()
pipeline = next(candidate / "pipeline" for candidate in [here, *here.parents] if (candidate / "Data_Lakehouse_lab" / "pipeline" / "transforms.py").exists())
sys.path.insert(0, str(pipeline))
from transforms import clean_trades

spark = SparkSession.builder.appName("shared-clean-trades").master("local[*]").getOrCreate()
rows = [
    ("T1", "vnm", "buy", "20", "10", "2026-11-17T08:00:00+00:00"),
    ("T1", "fpt", "buy", "99", "1", "2026-11-17T08:00:00+00:00"),
    ("LATE", "hpg", "buy", "10", "10", "2026-10-01T00:00:00+00:00"),
]
schema = "trade_id string, symbol string, side string, price string, quantity string, event_time string"
source = spark.createDataFrame(rows, schema=schema)

def batches(iterator):
    for frame in iterator:
        yield clean_trades(frame, as_of=pd.Timestamp("2026-11-17T09:00:00Z"))

out_schema = StructType([
    StructField("trade_id", StringType()),
    StructField("symbol", StringType()),
    StructField("side", StringType()),
    StructField("price", DoubleType()),
    StructField("quantity", LongType()),
    StructField("event_time", TimestampType()),
])
cleaned = source.mapInPandas(batches, out_schema)
cleaned.show(truncate=False)
assert cleaned.count() == 1
spark.stop()
